# E12 · Plantilla del laboratorio integrado
Copia este notebook a `soluciones/` con un nombre propio para no sobrescribir `E12_solucion.ipynb`. Completa las celdas, reinicia el kernel y ejecuta todo en orden. Consulta el [enunciado E12](../talleres/E12.md) y usa la solución docente como referencia después de intentar cada sección.

Elige PQRS o una ruta agroambiental. Conserva los originales y registra cualquier fallo pendiente. El producto es formativo; la entrega final permanece el 7 de noviembre.

## 1. Definir el producto
Enuncia la pregunta, unidad de observación, fuentes, período, granularidad, medida, salidas y límites de interpretación. En PQRS, un reporte no equivale a una persona única.

**Respuesta del equipo:**

**Pregunta:** ¿Cuántos reportes PQRS contiene cada período, por mes y municipio del afectado? Como control propio: ¿la muestra cubre los meses del semestre?

**Unidad de observación:** un reporte PQRS (fila con `OBJECTID`). Un reporte no equivale a una persona única.

**Fuentes:** muestras oficiales Supersalud (CC BY 4.0) `Muestra_1000_2023_II.csv`, `Muestra_1000_2024_I.csv`, `Muestra_1000_2024_II.csv`, verificadas con SHA-256 contra `kit/pqrs_fuentes.json`. Referencia: conteo oficial por mes del archivo completo (mismo manifiesto).

**Período:** 2023-II, 2024-I y 2024-II, desagregados por mes.

**Granularidad:** `corte × mes × municipio del afectado` (`afec_cod_mpio` a 5 dígitos; vacío = `SIN_INFORMAR`).

**Medida:** cantidad de reportes aceptados por grupo.

**Salidas:** Parquet de detalle, rechazados y agregado por ejecución; consulta DuckDB; `registro.json` de A y B; `publicacion_valida.json`; `cobertura_mensual.csv`; ficha de auditoría.

**Límites:** muestras de 1.000 reportes por corte; no estiman prevalencia ni volumen nacional. Al cruzar mes y municipio la mayoría de grupos tiene 1 o 2 reportes: no se deben comparar municipios. Las muestras son las primeras 1.000 filas de cada archivo, así que no representan todos los meses.

**Aportes del equipo sobre la solución de referencia:** (1) el MES en la granularidad; (2) dos reglas propias de coherencia con el corte (R5 mes dentro del semestre, R6 año del corte) con su propio caso de prueba; (3) consultas por mes, análisis de dispersión y comparación con el archivo completo; (4) el hallazgo de cobertura en la ficha de auditoría.

## 2. Preparar entorno y entradas
Identifica la raíz del repositorio, el kernel, las versiones de las bibliotecas, rutas de entrada y salida. Registra origen, período y SHA-256 de cada archivo; verifica disponibilidad y contrato. Utiliza Parquet y DuckDB con el entorno del curso.

In [1]:
# =====================================================================
# 2A · PREPARAR ENTORNO (antes del diagrama)
# [DIAGRAMA: preparación previa]
# Rutas, versiones y parámetros. Base: solución docente. EQUIPO: producto con MES y reglas v2.
# =====================================================================
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib
import json
import platform

import duckdb            # [DIAGRAMA: "DuckDB reconstruye desde el detalle"]
import pandas as pd
import pyarrow           # escribe y lee Parquet
from IPython.display import display


def encontrar_repositorio(inicio=None):
    inicio = Path.cwd() if inicio is None else Path(inicio)
    for candidato in (inicio, *inicio.parents):
        if (candidato / 'kit' / 'pqrs_fuentes.json').is_file():
            return candidato
        anidado = candidato / 'big-data-postgraduate'
        if (anidado / 'kit' / 'pqrs_fuentes.json').is_file():
            return anidado
    raise FileNotFoundError('No se encontró la raíz de big-data-postgraduate')


REPOSITORIO = encontrar_repositorio()
DATOS = REPOSITORIO / 'kit' / 'data' / 'pqrs_muestras'            # [DIAGRAMA: "3 muestras PQRS"]
SALIDA = REPOSITORIO / 'kit' / 'salidas' / 'soluciones' / 'E12_equipo_mes'   # EQUIPO: carpeta propia
MANIFIESTO_RUTA = REPOSITORIO / 'kit' / 'pqrs_fuentes.json'       # huellas, columnas y [DIAGRAMA: "Referencia oficial"]
SALIDA.mkdir(parents=True, exist_ok=True)

CONFIGURACION = {
    'producto': 'conteo_reportes_por_corte_mes_y_municipio',          # EQUIPO: pregunta con MES
    'version_reglas': 'e12-equipo-mes-v2',                          # EQUIPO: + 2 reglas propias
    'periodos': ['2023_II', '2024_I', '2024_II'],
    'dimensiones_agregado': ['corte_archivo', 'mes_num', 'territorio'],  # EQUIPO: período × MES × municipio
    'medida': 'reportes',
    'zona_horaria': 'America/Bogota',
    'responsable_incidencias': 'COMPLETAR: integrante que revisa rechazados y fallos',  # EQUIPO
}

print('Repositorio:', REPOSITORIO)
print('Datos:', DATOS)
print('Salidas:', SALIDA)
print('Python:', platform.python_version())
print('pandas:', pd.__version__, '| pyarrow:', pyarrow.__version__, '| duckdb:', duckdb.__version__)

Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Datos: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/data/pqrs_muestras
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/E12_equipo_mes
Python: 3.12.15
pandas: 2.2.3 | pyarrow: 19.0.1 | duckdb: 1.4.1


In [2]:
# =====================================================================
# 2B · HERRAMIENTAS DEL PIPELINE (solo se definen; se usan en las celdas siguientes)
# [DIAGRAMA: "Huella SHA-256 y contrato?", "Reglas", "Construir anio_mes y municipio", "agregado"]
# =====================================================================
# [DIAGRAMA: "Huella SHA-256?"] huella del ARCHIVO: si cambia un carácter, cambia toda la huella
def sha256_archivo(ruta, bloque=1024 * 1024):
    firma = hashlib.sha256()
    with Path(ruta).open('rb') as archivo:
        for fragmento in iter(lambda: archivo.read(bloque), b''):
            firma.update(fragmento)
    return firma.hexdigest()


# [DIAGRAMA: "Ejecucion B firmas A = B"] huella del CONTENIDO de una tabla (ordenada y como texto)
def firma_dataframe(datos, columnas, orden):
    normalizado = datos.loc[:, columnas].copy()
    normalizado = normalizado.sort_values(orden, kind='stable').reset_index(drop=True)
    normalizado = normalizado.astype('string').fillna('<NA>')
    texto = normalizado.to_csv(index=False, lineterminator='\n')
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()


# [DIAGRAMA: "contrato?" → "DETENER"] columnas con nombre y orden exactos; si no, se detiene
def validar_contrato(datos, columnas_esperadas, nombre):
    actuales = datos.columns.tolist()
    if actuales != columnas_esperadas:
        faltantes = sorted(set(columnas_esperadas) - set(actuales))
        extras = sorted(set(actuales) - set(columnas_esperadas))
        raise ValueError(
            f'Contrato incumplido en {nombre}; faltantes={faltantes}; extras={extras}; '
            'también debe conservarse el orden de columnas'
        )


# [DIAGRAMA: "rechazados.parquet con motivos"] acumula motivos: una fila rechazada se cuenta UNA vez
def agregar_motivo(serie, mascara, motivo):
    previo = serie.loc[mascara]
    serie.loc[mascara] = previo.where(previo.eq(''), previo + '|') + motivo


# [DIAGRAMA: "Reglas" → "Construir anio_mes y municipio" → "agregado periodo x mes x municipio"]
def validar_y_transformar(datos):
    trabajo = datos.copy()      # se trabaja sobre una copia: los originales no se tocan
    objectid = trabajo['OBJECTID'].astype('string').fillna('').str.strip()
    periodo_num = pd.to_numeric(trabajo['periodo'].astype('string').str.strip(), errors='coerce')
    mes_num = pd.to_numeric(trabajo['mes'].astype('string').str.strip(), errors='coerce')
    territorio_raw = trabajo['afec_cod_mpio'].astype('string').fillna('').str.strip()

    motivos = pd.Series('', index=trabajo.index, dtype='string')
    agregar_motivo(motivos, objectid.eq(''), 'OBJECTID_VACIO')
    agregar_motivo(motivos, periodo_num.isna() | periodo_num.mod(1).ne(0), 'PERIODO_INVALIDO')
    agregar_motivo(motivos, mes_num.isna() | mes_num.mod(1).ne(0) | ~mes_num.between(1, 12), 'MES_INVALIDO')
    territorio_invalido = territorio_raw.ne('') & ~territorio_raw.str.fullmatch(r'[0-9]{1,5}', na=False)
    agregar_motivo(motivos, territorio_invalido, 'TERRITORIO_INVALIDO')
    # --- REGLAS PROPIAS DEL EQUIPO: coherencia entre el contenido y el corte del archivo ---
    # R5: el corte I solo admite meses 1–6 y el corte II meses 7–12
    corte_I = trabajo['corte_archivo'].str.endswith('_I')
    en_semestre = (corte_I & mes_num.between(1, 6)) | (~corte_I & mes_num.between(7, 12))
    agregar_motivo(motivos, mes_num.notna() & ~en_semestre, 'MES_FUERA_DE_SEMESTRE')
    # R6: el año del reporte debe ser el año del corte (2024_I → 2024)
    anio_corte = pd.to_numeric(trabajo['corte_archivo'].str[:4], errors='coerce')
    agregar_motivo(motivos, periodo_num.notna() & periodo_num.ne(anio_corte), 'PERIODO_NO_COINCIDE_CORTE')

    rechazado = motivos.ne('')
    rechazados = trabajo.loc[rechazado, [
        'corte_archivo', 'archivo_fuente', 'fila_fuente', 'OBJECTID',
        'periodo', 'mes', 'afec_cod_mpio'
    ]].copy()
    rechazados['motivos_rechazo'] = motivos.loc[rechazado].to_numpy()

    aceptados = trabajo.loc[~rechazado].copy()
    aceptados['periodo_fuente'] = periodo_num.loc[~rechazado].astype('Int64').to_numpy()
    aceptados['mes_num'] = mes_num.loc[~rechazado].astype('Int64').to_numpy()
    territorio_aceptado = territorio_raw.loc[~rechazado]
    # [DIAGRAMA: "Construir ... municipio"] '5001' → '05001'; vacío → SIN_INFORMAR (se conserva)
    aceptados['territorio'] = territorio_aceptado.where(
        territorio_aceptado.ne(''), 'SIN_INFORMAR'
    ).str.zfill(5).to_numpy()
    aceptados.loc[aceptados['territorio'].eq('SIN_INFORMAR'), 'territorio'] = 'SIN_INFORMAR'
    aceptados['clave_reporte'] = (
        aceptados['corte_archivo'].astype(str) + '|' + aceptados['OBJECTID'].astype(str)
    )

    columnas_detalle = [
        'corte_archivo', 'archivo_fuente', 'fila_fuente', 'OBJECTID', 'clave_reporte',
        'periodo_fuente', 'mes_num', 'territorio', 'pqr_canal', 'ent_nombre'
    ]
    detalle = aceptados[columnas_detalle].copy()
    agregado = (
        detalle.groupby(['corte_archivo', 'mes_num', 'territorio'], dropna=False)   # EQUIPO: + MES
        .size().rename('reportes').reset_index()
        .sort_values(['corte_archivo', 'mes_num', 'territorio']).reset_index(drop=True)
    )
    return detalle, rechazados, agregado

In [3]:
# =====================================================================
# 2C · RECEPCIÓN DE LOS 3 ARCHIVOS REALES
# [DIAGRAMA: "3 muestras PQRS" → "Huella SHA-256 y contrato?" → "Leer como texto + corte, archivo, fila"]
# EVIDENCIA para comparar entre compañeros: las 3 huellas deben ser idénticas en todos los equipos.
# =====================================================================
with MANIFIESTO_RUTA.open(encoding='utf-8') as archivo:
    manifiesto = {item['periodo']: item for item in json.load(archivo)}

entradas = []
controles_entrada = []
for corte in CONFIGURACION['periodos']:
    referencia = manifiesto[corte]
    ruta = DATOS / f'Muestra_1000_{corte}.csv'
    huella = sha256_archivo(ruta)                       # [DIAGRAMA: "Huella SHA-256?"]
    assert huella == referencia['sha256_muestra'], f'Huella inesperada: {ruta.name}'

    datos = pd.read_csv(
        ruta, sep=referencia['separador'], encoding=referencia['codificacion'],
        dtype=str, keep_default_na=False
    )
    validar_contrato(datos, referencia['columnas'], ruta.name)   # [DIAGRAMA: "contrato?"]
    assert datos.shape == (referencia['filas_muestra'], referencia['cantidad_columnas'])

    # [DIAGRAMA: "+ corte, archivo, fila"] procedencia; corte_archivo = PERÍODO de la pregunta
    datos = datos.assign(
        corte_archivo=corte,
        archivo_fuente=ruta.name,
        fila_fuente=range(2, len(datos) + 2),
        sha256_fuente=huella,
    )
    entradas.append(datos)
    controles_entrada.append({
        'corte': corte, 'archivo': ruta.name, 'filas': len(datos),
        'columnas': referencia['cantidad_columnas'], 'sha256': huella
    })

df_entrada = pd.concat(entradas, ignore_index=True)
assert len(df_entrada) == sum(item['filas'] for item in controles_entrada) == 3000
assert df_entrada[['corte_archivo', 'OBJECTID']].duplicated().sum() == 0

display(pd.DataFrame(controles_entrada))
print('Filas recibidas:', len(df_entrada))

,corte,archivo,filas,columnas,sha256
0,2023_II,Muestra_1000_2023_II.csv,1000,38,5546bb62ca598654a821c147d1efdc92f165324c13f904...
1,2024_I,Muestra_1000_2024_I.csv,1000,38,f06abcc52537eff223894edbeb83aedd9719d45b7d0887...
2,2024_II,Muestra_1000_2024_II.csv,1000,38,38102044d2c66b86b1e081867bbc854c4ea220524b51a0...


Filas recibidas: 3000


## 3. Recepción y validación
Lee las entradas sin modificar los originales. Declara y aplica reglas de esquema, tipos y aceptación; separa aceptados y rechazados, conserva todos los motivos y cuenta cada fila rechazada una sola vez. Comprueba que leídas = aceptadas + rechazadas.

In [4]:
# =====================================================================
# 3A · CASO DIDÁCTICO DE REFERENCIA (6 filas, resultado conocido)
# [DIAGRAMA: "Reglas" → "rechazados" / "agregado" en miniatura]
# Con el MES en el agregado sigue dando 6 = 5 + 1 y 4 grupos (todas las filas 2024_I son de enero).
# =====================================================================
caso_seis_filas = pd.DataFrame({
    'corte_archivo': ['2024_I'] * 4 + ['2024_II'] * 2,
    'archivo_fuente': ['caso_didactico.csv'] * 6,
    'fila_fuente': range(2, 8),
    'OBJECTID': ['1', '2', '3', '4', '5', '6'],
    'periodo': ['2024', '2024', '2024', '2024', '2024', 'invalido'],
    'mes': ['1', '1', '1', '1', '7', '7'],
    'afec_cod_mpio': ['11001', '11001', '15001', '', '11001', '15001'],
    'pqr_canal': ['Web'] * 6,
    'ent_nombre': ['Entidad didáctica'] * 6,
})

detalle_caso, rechazados_caso, agregado_caso = validar_y_transformar(caso_seis_filas)
assert len(caso_seis_filas) == len(detalle_caso) + len(rechazados_caso) == 6
assert (len(detalle_caso), len(rechazados_caso), len(agregado_caso)) == (5, 1, 4)
assert rechazados_caso.iloc[0]['motivos_rechazo'] == 'PERIODO_INVALIDO'
assert agregado_caso['reportes'].sum() == 5

display(agregado_caso)
display(rechazados_caso[['fila_fuente', 'periodo', 'motivos_rechazo']])
print('Prueba superada: 6 leídas = 5 aceptadas + 1 rechazada; 4 grupos publicados')

,corte_archivo,mes_num,territorio,reportes
0,2024_I,1,11001,2
1,2024_I,1,15001,1
2,2024_I,1,SIN_INFORMAR,1
3,2024_II,7,11001,1


,fila_fuente,periodo,motivos_rechazo
5,7,invalido,PERIODO_INVALIDO


Prueba superada: 6 leídas = 5 aceptadas + 1 rechazada; 4 grupos publicados


In [5]:
# =====================================================================
# 3B · CASO PROPIO DEL EQUIPO: prueba de las reglas R5 y R6
# [DIAGRAMA: rombo "Reglas: mes, semestre y municipio"]
#   Fila 2: 2024_I, junio  → aceptada
#   Fila 3: 2024_I, agosto → rechazada MES_FUERA_DE_SEMESTRE (agosto no es del corte I)
#   Fila 4: 2024_II, julio, año 2023 → rechazada PERIODO_NO_COINCIDE_CORTE
#   Fila 5: 2024_II, marzo, año 2023 → DOS motivos, contada UNA vez
# Esperado: 4 = 1 aceptada + 3 rechazadas
# =====================================================================
caso_equipo = pd.DataFrame({
    'corte_archivo': ['2024_I', '2024_I', '2024_II', '2024_II'],
    'archivo_fuente': ['caso_equipo.csv'] * 4,
    'fila_fuente': range(2, 6),
    'OBJECTID': ['1', '2', '3', '4'],
    'periodo': ['2024', '2024', '2023', '2023'],
    'mes': ['6', '8', '7', '3'],
    'afec_cod_mpio': ['15238', '15001', '15759', '15238'],     # Duitama, Tunja, Sogamoso
    'pqr_canal': ['Web'] * 4,
    'ent_nombre': ['Entidad del equipo'] * 4,
})
det_eq, rech_eq, agr_eq = validar_y_transformar(caso_equipo)
display(rech_eq[['fila_fuente', 'corte_archivo', 'periodo', 'mes', 'motivos_rechazo']])

assert len(caso_equipo) == len(det_eq) + len(rech_eq) == 4
assert (len(det_eq), len(rech_eq)) == (1, 3)
assert rech_eq.set_index('fila_fuente')['motivos_rechazo'].to_dict() == {
    3: 'MES_FUERA_DE_SEMESTRE',
    4: 'PERIODO_NO_COINCIDE_CORTE',
    5: 'MES_FUERA_DE_SEMESTRE|PERIODO_NO_COINCIDE_CORTE'}
print('Reglas propias comprobadas: 4 = 1 + 3; la fila con dos motivos se cuenta una sola vez')

,fila_fuente,corte_archivo,periodo,mes,motivos_rechazo
1,3,2024_I,2024,8,MES_FUERA_DE_SEMESTRE
2,4,2024_II,2023,7,PERIODO_NO_COINCIDE_CORTE
3,5,2024_II,2023,3,MES_FUERA_DE_SEMESTRE|PERIODO_NO_COINCIDE_CORTE


Reglas propias comprobadas: 4 = 1 + 3; la fila con dos motivos se cuenta una sola vez


## 4. Transformación y agregado
Aplica las transformaciones necesarias y construye el detalle y el agregado con la granularidad definida. Si hay uniones, comprueba claves, cardinalidad y filas sin correspondencia. En la ruta agrícola conserva unidades y usa denominadores compatibles.

In [6]:
# =====================================================================
# 4 · TRANSFORMACIÓN Y AGREGADO
# [DIAGRAMA: "Construir anio_mes y municipio" → "agregado periodo x mes x municipio"]
# La transformación vive en validar_y_transformar (celda 2B). Aquí se muestra la evidencia:
# mes como entero, municipio a 5 dígitos, vacío como SIN_INFORMAR, agregado con MES.
# =====================================================================
display(caso_equipo)
print('Grupos en el caso:', len(agregado_caso), '| reportes:', agregado_caso['reportes'].sum())

# Evidencia de por qué se normaliza el municipio: en 2024_I hay códigos de 4 dígitos ('5001')
largo = df_entrada.assign(largo=df_entrada['afec_cod_mpio'].str.len()).groupby(['corte_archivo', 'largo']).size()
display(largo.unstack(fill_value=0))

,corte_archivo,archivo_fuente,fila_fuente,OBJECTID,periodo,mes,afec_cod_mpio,pqr_canal,ent_nombre
0,2024_I,caso_equipo.csv,2,1,2024,6,15238,Web,Entidad del equipo
1,2024_I,caso_equipo.csv,3,2,2024,8,15001,Web,Entidad del equipo
2,2024_II,caso_equipo.csv,4,3,2023,7,15759,Web,Entidad del equipo
3,2024_II,caso_equipo.csv,5,4,2023,3,15238,Web,Entidad del equipo


Grupos en el caso: 4 | reportes: 5


largo,4,5
corte_archivo,,
2023_II,0,1000
2024_I,186,814
2024_II,0,1000


## 5. Publicación candidata y consulta
Guarda detalle, rechazados y agregado en una carpeta identificada para la ejecución A. Relee los archivos y consulta con DuckDB. Compara el agregado reconstruido desde el detalle por cada grupo y por el total. Identifica la publicación como válida solo si supera los controles; conserva la publicación válida anterior ante un fallo.

In [7]:
# =====================================================================
# 5A · PIPELINE COMPLETO (columna derecha del diagrama)
# [DIAGRAMA: "agregado" → "DuckDB reconstruye desde el detalle" → "Concilia por grupo y total?" → "VALIDA registro.json"]
# Base: solución docente. EQUIPO: conciliación y firmas con el MES.
# =====================================================================
COLUMNAS_DETALLE = [
    'corte_archivo', 'archivo_fuente', 'fila_fuente', 'OBJECTID', 'clave_reporte',
    'periodo_fuente', 'mes_num', 'territorio', 'pqr_canal', 'ent_nombre'
]
COLUMNAS_RECHAZADOS = [
    'corte_archivo', 'archivo_fuente', 'fila_fuente', 'OBJECTID',
    'periodo', 'mes', 'afec_cod_mpio', 'motivos_rechazo'
]
COLUMNAS_AGREGADO = ['corte_archivo', 'mes_num', 'territorio', 'reportes']   # EQUIPO: + MES


def ejecutar_pipeline(identificador, datos_entrada, controles_fuente):
    carpeta = SALIDA / 'candidatos' / identificador
    carpeta.mkdir(parents=True, exist_ok=True)
    rutas = {
        'detalle': carpeta / 'detalle.parquet',
        'rechazados': carpeta / 'rechazados.parquet',
        'agregado': carpeta / 'agregado.parquet',
        'consulta': carpeta / 'consulta_duckdb.csv',
        'registro': carpeta / 'registro.json',
    }

    inicio = datetime.now(ZoneInfo(CONFIGURACION['zona_horaria']))
    detalle, rechazados, agregado = validar_y_transformar(datos_entrada)
    assert len(datos_entrada) == len(detalle) + len(rechazados)
    assert agregado['reportes'].sum() == len(detalle)

    # Publicación candidata en su propia carpeta (A, B...)
    detalle.to_parquet(rutas['detalle'], index=False, compression='zstd')
    rechazados.to_parquet(rutas['rechazados'], index=False, compression='zstd')
    agregado.to_parquet(rutas['agregado'], index=False, compression='zstd')

    # Se RELEE lo escrito: se verifica el archivo, no la memoria
    detalle_leido = pd.read_parquet(rutas['detalle'])
    rechazados_leidos = pd.read_parquet(rutas['rechazados'])
    agregado_leido = pd.read_parquet(rutas['agregado'])

    # [DIAGRAMA: "DuckDB reconstruye desde el detalle"] segundo camino de cálculo, en SQL
    with duckdb.connect() as conexion:
        consulta = conexion.execute(
            '''
            SELECT corte_archivo, mes_num, territorio, COUNT(*) AS reportes
            FROM read_parquet(?)
            GROUP BY corte_archivo, mes_num, territorio
            ORDER BY corte_archivo, mes_num, territorio
            ''',
            [str(rutas['detalle'])],
        ).df()
    consulta.to_csv(rutas['consulta'], index=False)

    # [DIAGRAMA: "Concilia por grupo y total?"] si no coincide, se detiene (no hay publicación)
    pd.testing.assert_frame_equal(
        agregado_leido[COLUMNAS_AGREGADO].reset_index(drop=True),
        consulta[COLUMNAS_AGREGADO].reset_index(drop=True),
        check_dtype=False,
    )
    assert len(datos_entrada) == len(detalle_leido) + len(rechazados_leidos)
    assert consulta['reportes'].sum() == len(detalle_leido)

    firmas = {
        'detalle': firma_dataframe(
            detalle_leido, COLUMNAS_DETALLE, ['corte_archivo', 'fila_fuente']
        ),
        'rechazados': firma_dataframe(
            rechazados_leidos, COLUMNAS_RECHAZADOS, ['corte_archivo', 'fila_fuente']
        ),
        'agregado': firma_dataframe(
            agregado_leido, COLUMNAS_AGREGADO, ['corte_archivo', 'mes_num', 'territorio']
        ),
    }
    fin = datetime.now(ZoneInfo(CONFIGURACION['zona_horaria']))
    # [DIAGRAMA: "VALIDA registro.json"] evidencia de auditoría de la ejecución
    registro = {
        'identificador': identificador,
        'estado': 'VALIDA',
        'producto': CONFIGURACION['producto'],
        'version_reglas': CONFIGURACION['version_reglas'],
        'inicio': inicio.isoformat(),
        'fin': fin.isoformat(),
        'entradas': controles_fuente,
        'conteos': {
            'leidas': len(datos_entrada),
            'aceptadas': len(detalle_leido),
            'rechazadas': len(rechazados_leidos),
            'grupos': len(agregado_leido),
            'total_agregado': int(agregado_leido['reportes'].sum()),
        },
        'controles': {
            'leidas_igual_aceptadas_mas_rechazadas': True,
            'agregado_igual_detalle_por_grupo': True,
            'agregado_igual_detalle_total': True,
            'originales_modificados': False,
        },
        'firmas_contenido': firmas,
        'rutas': {clave: str(valor.relative_to(REPOSITORIO)) for clave, valor in rutas.items()},
        'versiones': {
            'python': platform.python_version(), 'pandas': pd.__version__,
            'pyarrow': pyarrow.__version__, 'duckdb': duckdb.__version__,
        },
    }
    with rutas['registro'].open('w', encoding='utf-8') as archivo:
        json.dump(registro, archivo, ensure_ascii=False, indent=2)
    return registro, detalle_leido, rechazados_leidos, agregado_leido

In [8]:
# =====================================================================
# 5B · EJECUCIÓN A Y PRIMERA PUBLICACIÓN VÁLIDA
# [DIAGRAMA: "Concilia?" → "VALIDA registro.json"]
# =====================================================================
registro_a, detalle_a, rechazados_a, agregado_a = ejecutar_pipeline(
    'ejecucion_a', df_entrada, controles_entrada
)
assert registro_a['estado'] == 'VALIDA'

publicacion_ruta = SALIDA / 'publicacion_valida.json'
publicacion = {
    'ejecucion_valida': 'ejecucion_a',
    'estado': 'VALIDA',
    'actualizada': datetime.now(ZoneInfo(CONFIGURACION['zona_horaria'])).isoformat(),
    'motivo': 'Primera ejecución conciliada por grupo y total',
}
with publicacion_ruta.open('w', encoding='utf-8') as archivo:
    json.dump(publicacion, archivo, ensure_ascii=False, indent=2)

display(pd.DataFrame([registro_a['conteos']]))
display(agregado_a.head(10))
display(agregado_a.tail(10))
print('Publicación válida:', publicacion['ejecucion_valida'])

,leidas,aceptadas,rechazadas,grupos,total_agregado
0,3000,3000,0,781,3000


,corte_archivo,mes_num,territorio,reportes
0,2023_II,7,05001,85
1,2023_II,7,05034,1
2,2023_II,7,05040,1
3,2023_II,7,05045,2
4,2023_II,7,05051,1
5,2023_II,7,05079,1
6,2023_II,7,05088,18
7,2023_II,7,05129,4
8,2023_II,7,05142,1
9,2023_II,7,05154,2


,corte_archivo,mes_num,territorio,reportes
771,2024_II,7,81001,9
772,2024_II,7,81300,1
773,2024_II,7,85001,11
774,2024_II,7,85015,2
775,2024_II,7,85250,1
776,2024_II,7,86568,1
777,2024_II,7,86571,1
778,2024_II,7,86573,1
779,2024_II,7,86865,2
780,2024_II,7,91530,1


Publicación válida: ejecucion_a


In [9]:
# =====================================================================
# 5C · CONSULTAS PROPIAS DEL EQUIPO: responden la pregunta con el MES
# [DIAGRAMA: "agregado periodo x mes x municipio" → "HALLAZGO 1"; "Comparar mes" ← "Referencia oficial" → "HALLAZGO 2"]
# =====================================================================
carpeta_a = SALIDA / 'candidatos' / 'ejecucion_a'
with duckdb.connect() as con:
    con.execute(f"CREATE VIEW agregado AS SELECT * FROM read_parquet('{carpeta_a / 'agregado.parquet'}')")

    print('Q1 · Reportes por corte y mes (suma de todos los municipios)')
    display(con.execute('''SELECT corte_archivo, mes_num, SUM(reportes) AS reportes
                           FROM agregado GROUP BY corte_archivo, mes_num ORDER BY 1, 2''').df())

    print('Q2 · Los 10 grupos corte × mes × municipio con más reportes')
    display(con.execute('''SELECT corte_archivo, mes_num, territorio, reportes
                           FROM agregado ORDER BY reportes DESC, corte_archivo, mes_num LIMIT 10''').df())

    # [DIAGRAMA: "HALLAZGO 1"] ¿qué tan pocos reportes quedan por grupo?
    print('Q3 · Dispersión: ¿cuántos reportes tiene cada grupo?')
    display(con.execute('''SELECT corte_archivo, COUNT(*) AS grupos, ROUND(AVG(reportes), 1) AS promedio,
                                  SUM(CASE WHEN reportes = 1 THEN 1 ELSE 0 END) AS grupos_con_1,
                                  ROUND(100.0 * SUM(CASE WHEN reportes = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_con_1
                           FROM agregado GROUP BY corte_archivo ORDER BY 1''').df())

# [DIAGRAMA: "Referencia oficial" → "Comparar mes: muestra vs archivo completo" → "HALLAZGO 2"]
# Conteo oficial por mes del archivo completo (manifiesto) unido 1 a 1 con la muestra
referencia = pd.DataFrame([{'corte_archivo': c, 'mes_num': int(m), 'reportes_completo': n}
                           for c in CONFIGURACION['periodos'] for m, n in manifiesto[c]['meses'].items()])
muestra_mes = agregado_a.groupby(['corte_archivo', 'mes_num'], as_index=False)['reportes'].sum()
muestra_mes['mes_num'] = muestra_mes['mes_num'].astype(int)
cobertura = referencia.merge(muestra_mes, on=['corte_archivo', 'mes_num'], how='left', validate='one_to_one')
cobertura['reportes'] = cobertura['reportes'].fillna(0).astype(int)        # meses sin reportes en la muestra = 0
cobertura['pct_muestra'] = (100 * cobertura['reportes'] / cobertura.groupby('corte_archivo')['reportes'].transform('sum')).round(1)
cobertura['pct_completo'] = (100 * cobertura['reportes_completo'] / cobertura.groupby('corte_archivo')['reportes_completo'].transform('sum')).round(1)
assert cobertura['reportes'].sum() == len(detalle_a)                       # control: no se pierde ningún reporte
cobertura.to_csv(SALIDA / 'cobertura_mensual.csv', index=False)
print('Q4 · Muestra vs archivo completo por mes')
display(cobertura.sort_values(['corte_archivo', 'mes_num']).reset_index(drop=True))

# Evidencia del porqué: las muestras son las primeras filas de cada archivo
print('Rango de OBJECTID por corte:')
display(detalle_a.assign(oid=detalle_a['OBJECTID'].astype(int)).groupby('corte_archivo')['oid'].agg(['min', 'max']))

Q1 · Reportes por corte y mes (suma de todos los municipios)


,corte_archivo,mes_num,reportes
0,2023_II,7,1000.0
1,2024_I,1,11.0
2,2024_I,2,9.0
3,2024_I,3,17.0
4,2024_I,4,47.0
5,2024_I,5,188.0
6,2024_I,6,728.0
7,2024_II,7,1000.0


Q2 · Los 10 grupos corte × mes × municipio con más reportes


,corte_archivo,mes_num,territorio,reportes
0,2023_II,7,11001,230
1,2024_II,7,11001,205
2,2024_I,6,11001,148
3,2024_II,7,05001,98
4,2023_II,7,05001,85
5,2024_II,7,76001,85
6,2024_I,6,76001,63
7,2023_II,7,76001,61
8,2024_I,6,05001,47
9,2024_I,5,11001,42


Q3 · Dispersión: ¿cuántos reportes tiene cada grupo?


,corte_archivo,grupos,promedio,grupos_con_1,pct_con_1
0,2023_II,236,4.2,147.0,62.3
1,2024_I,329,3.0,206.0,62.6
2,2024_II,216,4.6,129.0,59.7


Q4 · Muestra vs archivo completo por mes


,corte_archivo,mes_num,reportes_completo,reportes,pct_muestra,pct_completo
0,2023_II,7,116031,1000,100.0,16.2
1,2023_II,8,129984,0,0.0,18.1
2,2023_II,9,128894,0,0.0,18.0
3,2023_II,10,136090,0,0.0,19.0
4,2023_II,11,122621,0,0.0,17.1
5,2023_II,12,83077,0,0.0,11.6
6,2024_I,1,110994,11,1.1,14.2
7,2024_I,2,120943,9,0.9,15.5
8,2024_I,3,119787,17,1.7,15.3
9,2024_I,4,154267,47,4.7,19.7


Rango de OBJECTID por corte:


,min,max
corte_archivo,,
2023_II,1,1000
2024_I,1,1000
2024_II,1,1000


## 6. Registro de ejecución
Registra identificador, fuentes y huellas, reglas o versión del código, parámetros, versiones de bibliotecas, conteos, controles, estado, errores y salida válida. Identifica responsable de revisar incidencias.

In [10]:
# =====================================================================
# 6 · REGISTRO DE LA EJECUCIÓN A
# [DIAGRAMA: "VALIDA registro.json"]
# =====================================================================
print('Estado A :', registro_a['estado'])
print('Conteos A:', registro_a['conteos'])
print('Responsable de incidencias:', CONFIGURACION['responsable_incidencias'])
display(pd.DataFrame(registro_a['entradas']))
print('Registro completo en:', SALIDA / 'candidatos' / 'ejecucion_a' / 'registro.json')

Estado A : VALIDA
Conteos A: {'leidas': 3000, 'aceptadas': 3000, 'rechazadas': 0, 'grupos': 781, 'total_agregado': 3000}
Responsable de incidencias: COMPLETAR: integrante que revisa rechazados y fallos


,corte,archivo,filas,columnas,sha256
0,2023_II,Muestra_1000_2023_II.csv,1000,38,5546bb62ca598654a821c147d1efdc92f165324c13f904...
1,2024_I,Muestra_1000_2024_I.csv,1000,38,f06abcc52537eff223894edbeb83aedd9719d45b7d0887...
2,2024_II,Muestra_1000_2024_II.csv,1000,38,38102044d2c66b86b1e081867bbc854c4ea220524b51a0...


Registro completo en: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/E12_equipo_mes/candidatos/ejecucion_a/registro.json


## 7. Reejecución y fallo controlado
Repite el recorrido con las mismas entradas y reglas en una carpeta B independiente. Compara contenido, claves y medidas de detalle, rechazados y agregado; los tiempos e identificadores pueden cambiar. Prueba un fallo de contrato sin alterar los originales y comprueba que no sustituye la publicación válida.

In [11]:
# =====================================================================
# 7A · EJECUCIÓN B: mismas entradas y reglas, otra carpeta
# [DIAGRAMA: "Ejecucion B firmas A = B"]
# Tiempos e identificador cambian; el CONTENIDO (firmas) debe ser idéntico.
# =====================================================================
registro_b, detalle_b, rechazados_b, agregado_b = ejecutar_pipeline(
    'ejecucion_b', df_entrada, controles_entrada
)

comparacion = {
    producto: registro_a['firmas_contenido'][producto] == registro_b['firmas_contenido'][producto]
    for producto in ['detalle', 'rechazados', 'agregado']
}
comparacion['conteos_iguales'] = registro_a['conteos'] == registro_b['conteos']
comparacion['estable'] = all(comparacion.values())
assert comparacion['estable'], comparacion

pd.testing.assert_frame_equal(detalle_a, detalle_b)
pd.testing.assert_frame_equal(rechazados_a, rechazados_b)
pd.testing.assert_frame_equal(agregado_a, agregado_b)

comparacion_documento = {
    'ejecucion_a': registro_a['identificador'],
    'ejecucion_b': registro_b['identificador'],
    'comparaciones': comparacion,
    'firmas_a': registro_a['firmas_contenido'],
    'firmas_b': registro_b['firmas_contenido'],
}
with (SALIDA / 'comparacion_reejecuciones.json').open('w', encoding='utf-8') as archivo:
    json.dump(comparacion_documento, archivo, ensure_ascii=False, indent=2)

# Solo ahora la ejecución B reemplaza a A como publicación identificada.
publicacion = {
    'ejecucion_valida': 'ejecucion_b',
    'ejecucion_anterior': 'ejecucion_a',
    'estado': 'VALIDA',
    'actualizada': datetime.now(ZoneInfo(CONFIGURACION['zona_horaria'])).isoformat(),
    'motivo': 'Segunda ejecución estable y conciliada; se conserva la ejecución anterior',
}
with publicacion_ruta.open('w', encoding='utf-8') as archivo:
    json.dump(publicacion, archivo, ensure_ascii=False, indent=2)

display(pd.DataFrame([comparacion]))
print('Publicación válida:', publicacion['ejecucion_valida'])

,detalle,rechazados,agregado,conteos_iguales,estable
0,True,True,True,True,True


Publicación válida: ejecucion_b


In [12]:
# =====================================================================
# 7B · FALLO CONTROLADO: falta la columna mes (solo en memoria)
# [DIAGRAMA: rama roja "FALLIDA → se conserva la anterior"]
# Esperado: el contrato detiene la entrada y la publicación válida no cambia.
# =====================================================================
entrada_defectuosa = entradas[0].drop(columns=['mes'])
try:
    validar_contrato(
        entrada_defectuosa.drop(columns=['corte_archivo', 'archivo_fuente', 'fila_fuente', 'sha256_fuente']),
        manifiesto['2023_II']['columnas'],
        'entrada_defectuosa_en_memoria',
    )
except ValueError as error:
    incidencia_controlada = {
        'estado': 'DETENIDA',
        'causa_observada': str(error),
        'accion': 'No publicar; restaurar desde el original verificado y volver a ejecutar',
        'publicacion_conservada': publicacion['ejecucion_valida'],
    }
else:
    raise AssertionError('El contrato debía detener la entrada sin la columna mes')

with (SALIDA / 'incidencia_contrato_controlada.json').open('w', encoding='utf-8') as archivo:
    json.dump(incidencia_controlada, archivo, ensure_ascii=False, indent=2)

display(incidencia_controlada)

{'estado': 'DETENIDA',
 'causa_observada': "Contrato incumplido en entrada_defectuosa_en_memoria; faltantes=['mes']; extras=[]; también debe conservarse el orden de columnas",
 'accion': 'No publicar; restaurar desde el original verificado y volver a ejecutar',
 'publicacion_conservada': 'ejecucion_b'}

## 8. Auditoría entre equipos
Otro equipo reproduce una consulta y revisa los controles. Registra quién revisó, evidencia, hallazgo o control satisfactorio, acción y comprobación posterior. Explica una limitación de interpretación.

In [13]:
# =====================================================================
# 8 · FICHA DE CONTROLES + CONSULTA PARA EL EQUIPO AUDITOR
# [DIAGRAMA: "Auditoria: otro equipo reproduce Bogota por mes"]
# EQUIPO: se agrega el control de cobertura mensual como HALLAZGO (no como incumplimiento).
# =====================================================================
auditoria = pd.DataFrame([
    {
        'control': 'Entradas identificadas', 'estado': 'CUMPLE',
        'evidencia': 'Tres archivos, cortes y SHA-256 en cada registro',
        'accion': 'Ninguna',
    },
    {
        'control': 'Conteos conciliados', 'estado': 'CUMPLE',
        'evidencia': f"{registro_b['conteos']['leidas']} = {registro_b['conteos']['aceptadas']} + {registro_b['conteos']['rechazadas']}",
        'accion': 'Ninguna',
    },
    {
        'control': 'Agregado contra detalle', 'estado': 'CUMPLE',
        'evidencia': 'DuckDB reprodujo cada grupo y el total del agregado',
        'accion': 'Ninguna',
    },
    {
        'control': 'Reejecución estable', 'estado': 'CUMPLE',
        'evidencia': 'Firmas de detalle, rechazados y agregado iguales entre A y B',
        'accion': 'Ninguna',
    },
    {
        'control': 'Límite de interpretación', 'estado': 'CUMPLE',
        'evidencia': 'Los conteos representan reportes de las muestras, no personas ni prevalencia',
        'accion': 'Mantener el alcance junto al producto',
    },
    {   # EQUIPO
        'control': 'Cobertura mensual de la muestra', 'estado': 'HALLAZGO',
        'evidencia': 'La muestra concentra julio (cortes II) y junio (2024_I); el archivo completo reparte 11–20 % por mes',
        'accion': 'No concluir tendencia mensual; usar archivos completos o muestra estratificada en E13',
    },
])
auditoria.to_csv(SALIDA / 'ficha_auditoria_referencia.csv', index=False)
display(auditoria)
assert auditoria.loc[auditoria['estado'] != 'HALLAZGO', 'estado'].eq('CUMPLE').all()

# Consulta que reproduce el equipo auditor: Bogotá (11001) por corte y mes desde el detalle de B
with duckdb.connect() as con:
    display(con.execute('''SELECT corte_archivo, mes_num, COUNT(*) AS reportes FROM read_parquet(?)
                           WHERE territorio = '11001' GROUP BY ALL ORDER BY 1, 2''',
                        [str(SALIDA / 'candidatos' / 'ejecucion_b' / 'detalle.parquet')]).df())

,control,estado,evidencia,accion
0,Entradas identificadas,CUMPLE,"Tres archivos, cortes y SHA-256 en cada registro",Ninguna
1,Conteos conciliados,CUMPLE,3000 = 3000 + 0,Ninguna
2,Agregado contra detalle,CUMPLE,DuckDB reprodujo cada grupo y el total del agr...,Ninguna
3,Reejecución estable,CUMPLE,"Firmas de detalle, rechazados y agregado igual...",Ninguna
4,Límite de interpretación,CUMPLE,Los conteos representan reportes de las muestr...,Mantener el alcance junto al producto
5,Cobertura mensual de la muestra,HALLAZGO,La muestra concentra julio (cortes II) y junio...,No concluir tendencia mensual; usar archivos c...


,corte_archivo,mes_num,reportes
0,2023_II,7,230
1,2024_I,1,4
2,2024_I,2,2
3,2024_I,3,3
4,2024_I,4,13
5,2024_I,5,42
6,2024_I,6,148
7,2024_II,7,205


**Ficha de auditoría (equipo auditor)**

| Campo | Registro |
|---|---|
| Equipo auditor | COMPLETAR |
| Consulta reproducida | Reportes de Bogotá (11001) por corte y mes desde `detalle.parquet` de la ejecución B |
| Resultado del auditor | COMPLETAR: ¿obtuvo los mismos conteos? |
| Controles revisados | leídas = aceptadas + rechazadas; DuckDB = agregado; firmas A = B; fallo sin publicar; reglas R5–R6 |
| Hallazgo o control satisfactorio | COMPLETAR |
| Acción y comprobación posterior | COMPLETAR |
| Limitación | La mayoría de grupos mes × municipio tiene 1 o 2 reportes y la muestra no cubre todos los meses. Un reporte no es una persona. |

## 9. Cierre y criterios de aceptación
Documenta evidencia para: ejecución completa; conteos conciliados; agregado consistente por grupo y total; y reejecución estable. Identifica pendientes y cómo regenerar las salidas. Entrega el notebook, manifiesto, consultas, registros de A y B y ficha de auditoría; las salidas voluminosas se entregan por el medio del curso y se excluyen de Git.

In [14]:
# =====================================================================
# 9 · CRITERIOS DE ACEPTACIÓN
# [DIAGRAMA: todo el recorrido]
# =====================================================================
criterios_finales = {
    'ejecucion_completa': registro_b['estado'] == 'VALIDA',
    'conteos_explicables': (
        registro_b['conteos']['leidas']
        == registro_b['conteos']['aceptadas'] + registro_b['conteos']['rechazadas']
    ),
    'agregado_consistente': registro_b['controles']['agregado_igual_detalle_por_grupo'],
    'reejecucion_estable': comparacion['estable'],
}
assert all(criterios_finales.values())

resumen_final = {
    'criterios': criterios_finales,
    'publicacion_valida': publicacion['ejecucion_valida'],
    'conteos': registro_b['conteos'],
    'salida_relativa': str(SALIDA.relative_to(REPOSITORIO)),
}
with (SALIDA / 'resumen_final.json').open('w', encoding='utf-8') as archivo:
    json.dump(resumen_final, archivo, ensure_ascii=False, indent=2)

display(pd.DataFrame([criterios_finales]))
display(pd.DataFrame([registro_b['conteos']]))
print('E12 completado y verificado. Salidas:', resumen_final['salida_relativa'])

,ejecucion_completa,conteos_explicables,agregado_consistente,reejecucion_estable
0,True,True,True,True


,leidas,aceptadas,rechazadas,grupos,total_agregado
0,3000,3000,0,781,3000


E12 completado y verificado. Salidas: kit/salidas/soluciones/E12_equipo_mes


**Conclusiones y pendientes:**

- **Pipeline:** 3.000 reportes con huella y contrato verificados; 3000 = 3000 + 0 con 6 reglas (4 de referencia + 2 del equipo), probadas con dos casos (6 = 5 + 1 y 4 = 1 + 3).
- **Por mes:** 2023-II y 2024-II solo tienen julio (1.000 cada uno); 2024-I: enero 11, febrero 9, marzo 17, abril 47, mayo 188, junio 728.
- **Por municipio:** Bogotá (11001) lidera (230 en julio 2023, 148 en junio 2024, 205 en julio 2024), seguida de Medellín (05001) y Cali (76001).
- **Hallazgo 1 (dispersión):** 781 grupos y cerca del 60 % con un solo reporte: no sirve para comparar municipios.
- **Hallazgo 2 (cobertura):** en el archivo completo cada mes pesa entre 11 % y 20 %; en la muestra julio es el 100 % en dos cortes y junio el 72,8 % en 2024-I. Son las primeras 1.000 filas (`OBJECTID` 1–1000), no una muestra aleatoria.
- **Propuesta E13:** usar los archivos completos o una muestra estratificada por mes.
- **Pendiente:** ficha del equipo auditor y responsable de incidencias.
- **Regenerar:** *Restart & Run All* con `.venv`. `kit/salidas/` no se sube a Git.